# A/B testing in R (draft notes)

## A/B testing workflow

1. **Define the objective**
   - Specify the business goal (e.g., conversion, retention)
   - Choose primary and secondary metrics
   - Determine the minimum detectable effect (MDE)
2. **Design the experiment**
   - Define treatment and control conditions
   - Specify inclusion criteria and sampling strategy
   - Perform power analysis to determine sample size
3. **Randomisation**
   - Randomly sample users from the target population
   - Randomly assign users to control and treatment groups
4. **Run and monitor**
   - Ensure data integrity and experiment validity
   - Check for sample ratio mismatch (SRM)
5. **Statistical inference**
   - Estimate treatment effects
   - Test for statistical significance
   - Report confidence intervals and effect sizes

## 1 Statistical framework

### Confusion matrix

| | predicted positive class | predicted negative class | |
| :-: | :-: | :-: | :-: |
| **actual positive class** | True positive<br>$(1-\beta)$ | False negative<br>$(\beta)$ | Sensitivity<br>$\left(\frac{TP}{TP+FN}\right)$ |
| **actual negative class** | False positive<br>$(\alpha)$ | True negative<br>$(1-\alpha)$ | Specificity<br>$\left(\frac{TN}{TN+FP}\right)$ |
| | Precision<br>$\left(\frac{TP}{TP+FP}\right)$ | Negative predictive value<br>$\left(\frac{TN}{TN+FN}\right)$ | Accuracy<br>$\left(\frac{TP+TN}{TP+TN+FP+FN}\right)$ |

### Family-wise error rate

Family-wise error rate captures the idea that as the number of hypothesis tests increases, so does the probability of observing at least one false positive. FWER is the probability of having at least one false positive across all tests, which grows with the number of tested variables, number of tests or fluctuation in the sample.

$$
FWER = 1 - (1-\alpha)^m
$$

Where 
- $\alpha$ denotes the probability of Type I Error (false positive),
- $m$ is the number of tests. 

## 2 Independent sample t-test

### t-test

An independent sample t-test assumes
- a random sample
- normally distributed data
- similar group variances

Assess assumptions
- Levene's test for variances
- Shapiro-Wilk or Jarque-Bera for normality
- histograms and QQ plots for the shape of the data

For a general hypothesis test $H_0: \theta = \theta_0$ under standard regularity conditions $W \overset{d}{\to}\mathcal{N}(0,1)$,

$$
W =\frac{\hat{\theta}-\theta_0}{SE(\hat{\theta})}, \qquad SE(\hat{\theta}) = \sqrt{\widehat{Var}(\hat{\theta})}
$$

#### Summary table

| test | statistic | distribution | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| one sample t-test | $t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}}$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1 \end{align*}$$ | $\bar{x}\pm t_{\alpha/2, df} \times \frac{s}{\sqrt{n}}$ |
| two-sample t-test<br>(equal variances) | $$\begin{align*} t&=\frac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}}, \\ \quad s_p^2&=\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2} \end{align*}$$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=n_1+n_2-2 \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}$ |
| two-sample t-test<br>(unequal variances) | $t=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=\frac{\left(\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}\right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1}+\frac{(s_2^2/n_2)^2}{n_2-1}} \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times \sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}$ |
| one sample z-test | $z=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ | $z\sim \mathcal{N}(0,1)$ | $\bar{x}\pm z_{\alpha/2} \times \frac{\sigma}{\sqrt{n}}$ |
| two-sample z-test | $z=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}}$ | $z\sim \mathcal{N}(0,1)$ | $(\bar{x}_1-\bar{x}_2)\pm z_{\alpha/2} \times \sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}$ |
| paired t-test | $$\begin{align*} t&=\frac{\bar{d}-0}{s_d/\sqrt{n}}, \\ \quad d_i&=x_{1i}-x_{2i} \end{align*}$$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1\end{align*}$$ | $\bar{d}\pm t_{\alpha/2, df} \times \frac{s_d}{\sqrt{n}}$ |

Recall that for $X\perp Y$, $Var(X-Y)=Var(X)+Var(Y)=Var(X+Y)$.

### Cohen's d

Effect size.

$$d = \frac{\bar{x}_1-\bar{x}_2}{s_{\text{pooled}}}$$

Where
- $s_{\text{pooled}}=\sqrt{\frac{s_1^2+s_2^2}{2}}$ for equal sample sizes,
- $s_{\text{pooled}}=\sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}$ for unequal sample sizes.

One-sample effect size uses an assumed mean, $d=\frac{\bar{x}-\mu}{s}$.

| d | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large | 

### Power

Power analysis allows us to compute the unknown, such as the required sample size.

To derive the formula, start with two [CLT](https://www.probabilitycourse.com/chapter7/7_1_2_central_limit_theorem.php) equations $Z_{\alpha/2}=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ and -$Z_{\beta}=\frac{\bar{x}-\mu_0+\delta}{\sigma/\sqrt{n}}$. Solve for $\bar{x}$, equate the expressions and solve for n, eliminating $\mu_0$ in the process.

$$\begin{align*}
n &= \frac{2\sigma^2 (Z_{\alpha/2} + Z_\beta)^2}{\delta^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{\left(\frac{\delta}{\sigma}\right)^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{d^2}
\end{align*}$$

Where 
- $\sigma$ is the unknown population standard deviation,
- $\alpha$ is the probability of a false positive (typically 0.05),
- $Z_{\alpha/2}$ the corresponding critical value for a two-tailed test (typically 1.96),
- $\beta$ is the probability of a false negative (typically 0.2),
- $Z_\beta$ the corresponding critical value (typically 0.84),
- $\delta$ is the effect size,
- $d$ the effect size in standard deviations (Cohen's d).

### R

``` R
# Test normality with Shapiro-Wilk
# H0: X ~ N
shapiro.test(x)

# Test normality with Jarque-Bera
# H0: X ~ N
tseries::jarque.bera.test(x)

# Assess group variances
# H0: var(x1) = var(x2)
car::leveneTest(x1 ~ x2, data = df)

# Compare means
# H0: mean(x1) = mean(x2)
stats::t.test(
	x1 ~ x2,
	data = df,
	alternative = "two.sided",
	var.equal = TRUE
)

# Cohen's d
effectsize::cohens_d(x1 ~ x2, data = df)

# Power analysis and sample size (comment out the desired variable)
pwr::pwr.t.test(
    #n = 200,
    d = .8, 
    power = .8, 
    sig.level = .05, 
    type = "two.sample", 
    alternative = "two.sided"
)
```

## 3 Proportions test

### z-test for proportions

Two-sample z-test compares proportions between groups. It assumes
- large sample ($np \geq 5$, $n(1-p) \geq 5$), 
- independent observations.

#### Summary table

| test | statistic | distribution | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| two-sample proportion test | $z=\frac{\hat p_e-\hat p_c}{\sqrt{\hat p_{pooled}(1-\hat p_{pooled})(\frac{1}{n_c}+\frac{1}{n_e})}}$ | $z\sim N(0,1)$ | $(\hat p_e-\hat p_c)\pm z_{\alpha/2}\sqrt{\frac{\hat p_e(1-\hat p_e)}{n_e}+\frac{\hat p_c(1-\hat p_c)}{n_c}}$ |

### Cohen's h

Effect size for proportions.

$$h = 2\arcsin{(\sqrt{p_1})} - 2\arcsin{(\sqrt{p_2})}$$

| h | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large |

### R

``` R
# Proportion test
# H0: p1 = p2
stats::prop.test()

# Power analysis (equal group sample sizes)
pwr::pwr.2p.test(
    #n = 200,
    h = .8, 
    power = .8, 
    sig.level = .05,
    alternative = "two.sided"
)

# Power analysis (unequal group sample sizes)
pwr::pwr.2p2n.test(
    #n1 = 200,
    n2 = 200,
    h = .8, 
    power = .8, 
    sig.level = .05,
    alternative = "two.sided"
)
```

## 4 Non-parametric test

Use the Mann-Whitney rank-sum test for non-normal data. Assign ranks to the measure and sum by group. Compute the z-score for U.

### Mann-Whitney U test

For two groups, we compute $U_1$ and $U_2$,

$$\begin{align*}
U_1 &= n_1n_2+\frac{n_1(n_1+1)}{2}-T_1 \\
U_2 &= n_1n_2+\frac{n_2(n_2+1)}{2}-T_2
\end{align*}$$

Where $T_k$ denotes the rank sum for group $k$, $n_k$ the number of cases in group $k$.

U-value:

$$U=\text{min}(U_1,U_2)$$

Expected value of U:

$$\mu_U=\frac{n_1n_2}{2}$$

Standard error:

$$\sigma_U=\sqrt{\frac{n_1n_2(n_1+n_2+1)}{12}}$$

z-score:

$$z=\frac{U-\mu_U}{\sigma_U}$$

### Effect size

| r | effect size |
| :-: | :-: |
| $\approx .1$ | small |
| $\approx .3$ | medium |
| $\approx .5$ | large |

``` R
# Mann-Whitney U test (rank sum difference)
# H0: P(X_1 > X_2) = 0.5 (equal distributions)
stats::wilcox.test(data ~ group, data = df)

# Effect size
effectsize::rank_biserial(data ~ group, data = df)

# Power analysis and sample size (comment out as appropriate)
MKpower::sim.ssize.wilcox.test()
```

## Critical and p values

``` R
# Two-tailed p-value from t-statistic
p_from_t <- function(t_value, df, tails = "two") {
    if (tails == "two") {
        return(2 * pt(-abs(t_value), df))
    } else if (tails == "lower") {
        return(pt(t_value, df, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pt(t_value, df, lower.tail = FALSE))
    }
}

# Get t-statistic from p-value
t_from_p <- function(p_value, df, tails = "two") {
    if (tails == "two") {
        return(qt(1 - p_value / 2, df))
    } else if (tails == "lower") {
        return(qt(p_value, df))
    } else if (tails == "upper") {
        return(qt(1 - p_value, df))
    }
}

# Two-tailed p-value from z-statistic
p_from_z <- function(z_value, tails = "two") {
    if (tails == "two") {
        return(2 * pnorm(-abs(z_value)))
    } else if (tails == "lower") {
        return(pnorm(z_value, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pnorm(z_value, lower.tail = FALSE))
    }
}

# Get z-statistic from p-value
z_from_p <- function(p_value, tails = "two") {
    if (tails == "two") {
        return(qnorm(1 - p_value / 2))
    } else if (tails == "lower") {
        return(qnorm(p_value))
    } else if (tails == "upper") {
        return(qnorm(1 - p_value))
    }
}
```

In [1]:
# Libraries
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Power and sample size

In [2]:
## Compute the required sample size n
pwr::pwr.t.test(d = .8, power = 0.8, sig.level = 0.05, type = "one.sample", alternative = "two.sided")


     One-sample t test power calculation 

              n = 14.30276
              d = 0.8
      sig.level = 0.05
          power = 0.8
    alternative = two.sided
